### Delay

Python port of `notebooks/ts/Delay.ipynb`. This notebook uses the shared `max_plus.py` operators and writes generated visualization HTML under `notebooks/html/`.

Note: the actual delay tested might not be 100 ms.

<img src="../../img/Delay100ms.svg" alt="drawing" width="500"/>

In [ ]:
import heapq
from dataclasses import dataclass, field

import numpy as np

from max_plus import EPS, INF, oplus, otimes, reset
from timeline import write_timeline_to_file

print("Max-plus algebra and timeline helpers imported successfully")

In [ ]:
# System Max-Plus Encoding

# Create state vector for the system.
# x[0][i] is the earliest physical firing time computed for reaction i.
x = np.array([[EPS, EPS]])

# Indices for system components.
a = 0  # Reaction A index
b = 1  # Reaction B index

print("System setup complete")

In [ ]:
# System Parameters

# Logical time. t_bar blasts the scalar logical time into a row vector so it can
# participate in the same max-plus matrix equation as x.
t = EPS
t_bar = np.array([[t, t]])

# Timer parameters.
offset = 0
period = 1000  # Unit is ms. A period of 150ms is too small; barrier sync dominates.
timer_value = offset

# Delay parameter.
delay = 500  # A delay < 200 ms should have no effect on firing time because of barrier sync.

# Maxwait for decentralized coordination. INF disables the bound.
maxwait_a = INF
maxwait_b = INF

# Execution times. A zero in e means that reaction is skipped for the tag.
e = None
exact_exec_time_a = None  # 1 ms
exact_exec_time_b = None  # 101 ms normally, 1120 ms under the injected fault

# Identity matrix for max-plus algebra.
identity = np.array([
    [0, EPS],
    [EPS, 0],
])

# System matrices.
gamma = None       # Dependency matrix: gamma[i][j] is dependency from reaction j to reaction i.
gamma_star = None  # Kleene star of gamma; apparent latency across dependency paths.
B = None           # Barrier synchronization matrix.

print("System parameters initialized")

In [ ]:
@dataclass(order=True)
class Event:
    time: float
    order: int
    trigger: str = field(compare=False)


event_q = []
event_counter = 0


def enqueue_event(time, trigger):
    global event_counter
    heapq.heappush(event_q, Event(time, event_counter, trigger))
    event_counter += 1


def print_queue(event_q):
    print(f"Event Queue Contents ({len(event_q)} events):")
    for idx, event in enumerate(sorted(event_q)):
        print(f"  [{idx}] time: {event.time}, trigger: {event.trigger}")


def update_apparent_latencies(e_prime):
    global e, gamma, gamma_star, B
    e = e_prime

    # Update dependency matrix.
    # Reaction A depends on itself with zero latency and has no dependency on B.
    # Reaction B depends on A with A's apparent execution time and on itself with zero latency.
    gamma = np.array([
        [0, EPS],
        [e_prime[0][a], 0],
    ])

    # For this 2x2 dependency matrix, Gamma* reduces to I oplus Gamma when
    # there is no circuit with positive weight.
    gamma_star = oplus(identity, gamma)

    # Barrier synchronization matrix. Each row contains the reactions' execution times.
    B = np.array([
        [e_prime[0][a], e_prime[0][b]],
        [e_prime[0][a], e_prime[0][b]],
    ])

    print("e:", e_prime)
    print("Gamma:", gamma)
    print("GammaStar:", gamma_star)
    print("B:", B)

In [ ]:
# System evolution in factored max-plus form:
# x' = (GammaStar otimes B) otimes x oplus (GammaStar otimes tBar)
def step(x, t_bar):
    x_t = x.T
    t_t = t_bar.T
    t_prime = t_bar[0][a]

    # Dependency closure followed by barrier synchronization.
    GB = otimes(gamma_star, B)
    print("GB:", GB)

    # Propagate previous physical firing times.
    GBx = otimes(GB, x_t)
    print("GBx:", GBx)

    # Propagate the current logical tag through the dependency closure.
    Gt = otimes(gamma_star, t_t)
    print("Gt:", Gt)

    result = oplus(GBx, Gt).T

    # Apply decentralized maxwait constraints. With maxwait = INF, this does not
    # change the result, but the min form is kept to match the semantic model.
    return np.array([[
        min(maxwait_a + t_prime, result[0][a]),
        min(maxwait_b + t_prime, result[0][b]),
    ]])


def report_stats(k, x, t):
    lag_k = x[0] - t
    print(f"Tag index k={k}")
    print(f"t(k)= {t}")
    print(f"x(k) = earliest possible firing times = {x[0]}")
    print(f"lag(k) = {lag_k}")


print("System functions defined")

In [ ]:
# System Simulation Setup

t = 0
t_bar = np.array([[t, t]])

# Reset x to eps using the shared helper.
x = reset(x)

k = 0

print("System simulation ready")

### Simulation

Run the following cell to step through the queue-driven Delay simulation and collect data for visualization.

In [ ]:
x_arr = []
t_arr = []
e_arr = []

# Fault injection counter.
count = 0

# Reinitialize the queue so rerunning this cell starts from a clean simulation state.
event_q.clear()
event_counter = 0

# Add initial events.
enqueue_event(offset, "timer")
enqueue_event(offset, "inB")

# Define initial execution times.
exact_exec_time_a = 1
exact_exec_time_b = 101
e = np.array([[exact_exec_time_a, exact_exec_time_b]])
update_apparent_latencies(e)

simulation_steps = 20

for k_prime in range(simulation_steps):
    print_queue(event_q)

    # Pop all events at the earliest logical time.
    t_prime = event_q[0].time if event_q else INF
    t = t_prime
    t_bar = np.array([[t_prime, t_prime]])

    # Step the system from k to k+1 using max-plus operators.
    x_prime = step(x, t_bar)
    x_arr.append(x_prime.copy())
    t_arr.append(t_prime)

    print("Iteration k' =", k_prime)
    print("Logical time t' =", t_prime)
    print("State x' =", x_prime)
    print("Lag = [", ", ".join(f"{xi - t_prime:.3f}" for xi in x_prime[0]), "]")

    # Trigger fault.
    if count == 2:
        count = 0
        exact_exec_time_b = 1120
    else:
        count += 1
        exact_exec_time_b = 101

    x = x_prime

    events = []
    while event_q and event_q[0].time == t_prime:
        events.append(heapq.heappop(event_q))

    # Update apparent latencies by inspecting popped events.
    # Push new events into the queue based on popped events.
    e_prime = np.array([[0, 0]])
    for event in events:
        if event.trigger == "timer":
            e_prime[0][a] = exact_exec_time_a
            # The timer and inB triggers are semantically simultaneous because
            # there are zero delays from A to B, so enqueue both.
            enqueue_event(t_prime + period, "timer")
            enqueue_event(t_prime + period, "inB")
        elif event.trigger == "inA":
            e_prime[0][a] = exact_exec_time_a
        elif event.trigger == "inB":
            e_prime[0][b] = exact_exec_time_b
            enqueue_event(t_prime + delay, "inA")

    e_arr.append(e_prime[0].copy())
    update_apparent_latencies(e_prime)
    print_queue(event_q)

    print("=" * 50)

print("\n" + "=" * 60)
print("SIMULATION COMPLETE")
print("Final results stored in x_arr, t_arr, e_arr")

## HTML Timeline Export

Generate a Vis.js timeline and write it under `notebooks/html/`.

In [ ]:
write_timeline_to_file(
    x_arr,
    t_arr,
    e_arr,
    reaction_names=["A", "B"],
    filename="../html/delay-timeline.html",
    title="Delay Timeline",
    subtitle="Earliest reaction firing times for the Delay max-plus model.",
)